# Data Exploration - RetailRocket Dataset

This notebook explores the preprocessed RetailRocket e-commerce dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

## 1. Load Processed Data

In [ ]:
# Load dataset statistics
stats_df = pd.read_parquet('../data/processed/dataset_stats.parquet')
print("Dataset Statistics:")
for col in stats_df.columns:
    print(f"  {col}: {stats_df[col].values[0]}")

In [ ]:
# Load interactions
train_df = pd.read_parquet('../data/processed/train_interactions.parquet')
val_df = pd.read_parquet('../data/processed/val_interactions.parquet')
test_df = pd.read_parquet('../data/processed/test_interactions.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
# Load item features
item_features = pd.read_parquet('../data/processed/item_features.parquet')
print(f"Item features shape: {item_features.shape}")
item_features.head()

In [ ]:
# Load user sequences
train_sequences = pd.read_parquet('../data/processed/train_sequences.parquet')
print(f"User sequences shape: {train_sequences.shape}")
train_sequences.head()

## 2. Interaction Analysis

In [ ]:
# Event type distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (df, title) in enumerate([(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]):
    event_counts = df['event'].value_counts()
    axes[idx].pie(event_counts.values, labels=event_counts.index, autopct='%1.1f%%')
    axes[idx].set_title(f'{title} - Event Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# User interaction distribution
user_interactions = train_df.groupby('user_idx').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_interactions, bins=50, edgecolor='black')
axes[0].set_xlabel('Number of interactions')
axes[0].set_ylabel('Number of users')
axes[0].set_title('User Interaction Distribution')
axes[0].set_yscale('log')

axes[1].boxplot(user_interactions)
axes[1].set_ylabel('Number of interactions')
axes[1].set_title('User Interaction Boxplot')

plt.tight_layout()
plt.show()

print(f"Mean interactions per user: {user_interactions.mean():.2f}")
print(f"Median interactions per user: {user_interactions.median():.0f}")
print(f"Max interactions per user: {user_interactions.max()}")

In [ ]:
# Item popularity distribution
item_interactions = train_df.groupby('item_idx').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(item_interactions, bins=50, edgecolor='black')
axes[0].set_xlabel('Number of interactions')
axes[0].set_ylabel('Number of items')
axes[0].set_title('Item Popularity Distribution')
axes[0].set_yscale('log')

# Top 20 most popular items
top_items = item_interactions.nlargest(20)
axes[1].barh(range(len(top_items)), top_items.values)
axes[1].set_xlabel('Number of interactions')
axes[1].set_ylabel('Item rank')
axes[1].set_title('Top 20 Most Popular Items')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"Mean interactions per item: {item_interactions.mean():.2f}")
print(f"Median interactions per item: {item_interactions.median():.0f}")
print(f"Max interactions per item: {item_interactions.max()}")

## 3. Item Feature Analysis

In [ ]:
# Category distribution
category_counts = item_features[item_features['categoryid'] != -1]['categoryid'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top categories
top_categories = category_counts.head(20)
axes[0].barh(range(len(top_categories)), top_categories.values)
axes[0].set_xlabel('Number of items')
axes[0].set_ylabel('Category ID')
axes[0].set_title('Top 20 Categories by Item Count')
axes[0].invert_yaxis()

# Category size distribution
axes[1].hist(category_counts, bins=30, edgecolor='black')
axes[1].set_xlabel('Number of items in category')
axes[1].set_ylabel('Number of categories')
axes[1].set_title('Category Size Distribution')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Total unique categories: {len(category_counts)}")
print(f"Items without category: {(item_features['categoryid'] == -1).sum()}")

In [ ]:
# Price distribution
prices = item_features['price'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(prices, bins=50, edgecolor='black')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Number of items')
axes[0].set_title('Price Distribution')

# Log scale
axes[1].hist(np.log1p(prices), bins=50, edgecolor='black')
axes[1].set_xlabel('Log(Price + 1)')
axes[1].set_ylabel('Number of items')
axes[1].set_title('Price Distribution (Log Scale)')

plt.tight_layout()
plt.show()

print(f"Price statistics:")
print(prices.describe())

In [ ]:
# Price bucket distribution
price_bucket_counts = item_features['price_bucket'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(price_bucket_counts.index, price_bucket_counts.values)
plt.xlabel('Price Bucket')
plt.ylabel('Number of Items')
plt.title('Price Bucket Distribution')
plt.xticks(price_bucket_counts.index)
plt.show()

print("Price bucket mapping:")
print("  0: [0, 1000)")
print("  1: [1000, 5000)")
print("  2: [5000, 10000)")
print("  3: [10000, 50000)")
print("  4: [50000, ∞)")
print(" -1: Unknown")

## 4. Sequence Analysis

In [ ]:
# Sequence length distribution
seq_lengths = train_sequences['sequence_length']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(seq_lengths, bins=50, edgecolor='black')
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('User Sequence Length Distribution')

axes[1].boxplot(seq_lengths)
axes[1].set_ylabel('Sequence Length')
axes[1].set_title('Sequence Length Boxplot')

plt.tight_layout()
plt.show()

print(f"Sequence length statistics:")
print(seq_lengths.describe())

## 5. Temporal Analysis

In [ ]:
# Interactions over time
train_df['date'] = pd.to_datetime(train_df['datetime']).dt.date
daily_interactions = train_df.groupby('date').size()

plt.figure(figsize=(14, 5))
plt.plot(daily_interactions.index, daily_interactions.values)
plt.xlabel('Date')
plt.ylabel('Number of Interactions')
plt.title('Daily Interactions Over Time (Train Set)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Event types over time
event_by_date = train_df.groupby(['date', 'event']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Absolute counts
event_by_date.plot(ax=axes[0], marker='o')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Number of Events')
axes[0].set_title('Events Over Time (Absolute)')
axes[0].legend(title='Event Type')
axes[0].tick_params(axis='x', rotation=45)

# Relative proportions
event_by_date_pct = event_by_date.div(event_by_date.sum(axis=1), axis=0) * 100
event_by_date_pct.plot(ax=axes[1], marker='o')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Percentage of Events')
axes[1].set_title('Events Over Time (Relative %)')
axes[1].legend(title='Event Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Data Sparsity Analysis

In [ ]:
# Calculate sparsity
num_users = train_df['user_idx'].nunique()
num_items = train_df['item_idx'].nunique()
num_interactions = len(train_df)

total_possible = num_users * num_items
sparsity = 1 - (num_interactions / total_possible)

print(f"Data Sparsity Analysis:")
print(f"  Number of users: {num_users:,}")
print(f"  Number of items: {num_items:,}")
print(f"  Number of interactions: {num_interactions:,}")
print(f"  Total possible interactions: {total_possible:,}")
print(f"  Sparsity: {sparsity*100:.4f}%")
print(f"  Density: {(1-sparsity)*100:.4f}%")

## 7. Summary

Key insights from the data exploration:
- Dataset is highly sparse (typical for recommendation systems)
- View events dominate (~96%), followed by addtocart and transaction
- Long-tail distribution for both users and items
- Price distribution is right-skewed
- Most users have short interaction sequences